# 2. Oxygen adsorption on Cu(111)

In this exercise we are going to leverage some pre-built tools in the Atomic Simulation Environment (ASE) package. In particular, we are going to use ASE's build module to construct a Cu(111) surface slab, and then we are going to perform structural optimizations for various oxygen adsorption sites using the MACE-MP Machine-Learned Interatomic Potential. We will evaluate the binding energy at these adsorption sites and compare them to an experimental value.

One advantage in constructing our surface using ase.build.fcc111 is that adsorption sites are already pre-built into the object. It is then trivial to add a single adsorbate oxygen atom with ase.build.add_adsorbate. The alternative is to construct the ase.Atoms object manually, and then manually placing adsorbate oxygen atoms.

The goal of this exercise is to gain experience using the ASE package with Machine-Learned Interatomic Potentials. AIMNet2, MACE, and UMA have interfaces to ASE.

**Kernel:** MACE.
Run cells from top to bottom in a fresh kernel. Each setup creates a new results directory.

**Learning goals:** build a slab; constrain substrate layers; create a MACE calculator; optimize structures; compare adsorption geometries and energies

**Working pattern:** predict a result, run the calculation, inspect the geometry and convergence,
Energy is reported in eV, length in angstrom, and force in eV/angstrom.


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0,str(Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.visualize import view
from demo_tools.helpers import start_exercise

_, OUTPUT = start_exercise()


Python: /home/nchopper/anaconda3/envs/mace_demo/bin/python
Inputs: /home/nchopper/mlip-demo/workshop_demo/InorganicCrystals
New results: /home/nchopper/mlip-demo/workshop_demo/InorganicCrystals/results/20260923-162852-5b2c54


## 1. Build the substrate
We use a small slab for workshop speed, not a converged surface model. The bottom layer is fixed;
the other atoms can relax. All site calculations use the same cell, vacuum, and constraints.
Predict the preferred site: ontop, bridge, fcc, or hcp. A starting site may change during relaxation.

For this exercise, we will be using the Atomic Simulation Environment's (ASE) built-in builder tools. fcc111 returns an ase.Atoms object for a face-centered cubic cell with the (111) miller indice surface


In [12]:
from ase.build import fcc111, add_adsorbate
from ase.constraints import FixAtoms
from ase.optimize import BFGS

# Create a Cu(111) slab with 3 layers and a vacuum of 10 Å using ASE's internal default lattice spacing for Cu
slab = fcc111('Cu', size=(2, 2, 3), vacuum=10.0)

# Fix the bottom layer of the slab to simulate a surface with a bulk-like region
#   slab.get_tags() returns an array of integers representing the layer index of each atom in the slab.
#   The bottom layer is identified as the layer with the maximum tag value.
#      So slab.get_tags() == slab.get_tags().max() creates a boolean array 
#      where True corresponds to atoms in the bottom layer.
slab.set_constraint(FixAtoms(mask=slab.get_tags() == slab.get_tags().max()))

view(slab, viewer='x3d')

## 2. Create our Cu(111)+O systems

In [49]:
sites = ['ontop', 'bridge', 'fcc', 'hcp']
systems = {}
for site in sites:
    system = slab.copy()
    add_adsorbate(system, 'O', height=1.5, position=site)
    # Reset the constraint after adding the adsorbate.
    system.set_constraint(FixAtoms(mask = slab.get_tags() == slab.get_tags().max()))
    systems[site] = {'pre-optimized': system}

Visualize our systems

In [50]:
view(systems['ontop']['pre-optimized'], viewer='x3d')

In [51]:
view(systems['bridge']['pre-optimized'], viewer='x3d')

In [52]:
view(systems['fcc']['pre-optimized'], viewer='x3d')

In [53]:
view(systems['hcp']['pre-optimized'], viewer='x3d')

## 2. Create our MACE calculator object

To optimize our structures, we have to create a calculator object and attach it to the ase.Atoms objects.  The calculator will evaluate energies and forces.

In [ ]:
from mace.calculators import MACECalculator
DEVICE = 'cpu'  # Only select cuda inside a GPU allocation with a compatible environment.
DEFAULT_DTYPE = 'float64'

# * MACE-MP medium: '../models/2023-12-03-mace-128-L1_epoch-199.model'
# MACE-MP small: '../models/2023-12-10-mace-128-L0_energy_epoch-249.model'
# More models can be found here https://github.com/ACEsuit/mace-foundations
calculator = MACECalculator(model_paths='../models/2023-12-03-mace-128-L1_epoch-199.model',
                            device=DEVICE, default_dtype=DEFAULT_DTYPE)


/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/home/nchopper/anaconda3/envs/mace_demo/lib/python3.12/site-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


## 3. Setup and perform relaxation

In [55]:
from ase.optimize import BFGS
for site in sites:
    
    # Get energy before optimization
    systems[site]['pre-optimized'].calc = calculator
    systems[site]['preoptenergy'] = systems[site]['pre-optimized'].get_potential_energy()

    # Get a copy of the pre-optimized system and assign the calculator to it.
    systems[site]['optimized'] = systems[site]['pre-optimized'].copy()
    systems[site]['optimized'].calc = calculator

    # Optimize the geometry of the system using the BFGS algorithm. 
    dyn = BFGS(systems[site]['optimized']).run(fmax=0.05, steps=150)
    systems[site]['optenergy'] = systems[site]['optimized'].get_potential_energy()



      Step     Time          Energy          fmax
BFGS:    0 12:15:07      -48.838193       11.163249
BFGS:    1 12:15:07      -49.929572        0.946350
BFGS:    2 12:15:08      -49.947387        0.666878
BFGS:    3 12:15:08      -49.961799        0.358315
BFGS:    4 12:15:08      -49.963907        0.142945
BFGS:    5 12:15:09      -49.965193        0.080462
BFGS:    6 12:15:09      -49.968008        0.112540
BFGS:    7 12:15:09      -49.968632        0.073955
BFGS:    8 12:15:10      -49.969320        0.077781
BFGS:    9 12:15:10      -49.970148        0.083386
BFGS:   10 12:15:10      -49.971280        0.085408
BFGS:   11 12:15:11      -49.972088        0.059135
BFGS:   12 12:15:11      -49.972449        0.048327
      Step     Time          Energy          fmax
BFGS:    0 12:15:11      -51.199798        1.781527
BFGS:    1 12:15:12      -51.254624        1.447353
BFGS:    2 12:15:12      -51.351148        0.978273
BFGS:    3 12:15:12      -51.371124        0.645147
BFGS:    4 12:15

### Inpect geometries

In [64]:
site_to_inspect = 'bridge'
view(systems[site_to_inspect]['optimized'], viewer='x3d')

# !!! Notice that the bridge site optimized to the fcc site !!! #



### Ranking based on absolute energy

These are not binding energies!

In [65]:
# Ranking the adsorption sites
# We will rank the adsorption sites based on their optimized energies. Lower energy indicates a more stable adsorption site.
site_energies = {site: systems[site]['optenergy'] for site in sites}
ranked_sites = sorted(site_energies, key=site_energies.get)

site_energies_preop = {site: systems[site]['preoptenergy'] for site in sites}
ranked_sites_preop = sorted(site_energies_preop, key=site_energies_preop.get)

print('Most stable to least stable')
for site in ranked_sites:
    print(f"{site}: {site_energies[site]}")

print('Most stable to least stable (pre-optimization)')
for site in ranked_sites_preop:
    print(f"{site}: {site_energies_preop[site]}")

Most stable to least stable
fcc: -51.82944321816795
bridge: -51.82485174002644
hcp: -51.8128436183296
ontop: -49.97244931492864
Most stable to least stable (pre-optimization)
fcc: -51.40230997999212
hcp: -51.39129486424219
bridge: -51.19979761367664
ontop: -48.838192659515755


## 4. Adsorption energies
The primary comparison in the previous cell is between relaxed cells with **identical composition**. Their energy
differences do not require an isolated-oxygen reference.

An adsorption energy additionally needs a physically appropriate reference:

$$E_{ads}=E_{slab+O}-E_{slab}-E_{reference}.$$

Atomic O and half an O2 molecule define different quantities. We will use Atomic O and a relaxed, clean slab for our references.


In [72]:
# Setup clean Cu(111) slab
from ase.build import fcc111
cu_slab = fcc111('Cu', size=(2, 2, 3), vacuum=10.0)

# Setup oxygen atom
from ase import Atoms
oxygen = Atoms('O')

# Optimize the slab and get its energy
cu_slab.calc = calculator
dyn = BFGS(cu_slab)
dyn.run(fmax=0.05)
cu_slab_energy = cu_slab.get_potential_energy()

# Calculate atomic oxygen energy (no need to optimize since it is a single atom)
oxygen.calc = calculator
oxygen_energy = oxygen.get_potential_energy()

      Step     Time          Energy          fmax
BFGS:    0 12:35:18      -45.351450        0.034923


In [76]:
# Binding energies
for site in sites:
    systems[site]['E_ads'] = systems[site]['optenergy'] - cu_slab_energy - oxygen_energy

adsorption_energies = {site: systems[site]['E_ads'] for site in sites}
ranked_adsorption_energies = sorted(adsorption_energies.items(), key=lambda x: x[1])
for site, energy in ranked_adsorption_energies:
    print(f"Site: {site}, Adsorption Energy: {energy}")


Site: fcc, Adsorption Energy: -4.39653260284198
Site: bridge, Adsorption Energy: -4.391941124700467
Site: hcp, Adsorption Energy: -4.37993300300363
Site: ontop, Adsorption Energy: -2.5395386996026668



Experimental value

E. Shustorovich and A. T. Bell, Surf. Sci. 268, 397 (1992).https://doi.org/10.1016/0039-6028(92)90979-G'

Adsorption energy: -4.47 eV

## Review